In [15]:
#Data Collection
import pandas as pd

df = pd.read_csv("Dataset/default of credit card clients.csv")

print(df.head())
print(df.shape)
print(df.info())

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_1  PAY_2  PAY_3  PAY_4  \
0   1      20000    2          2         1   24      2      2     -1     -1   
1   2     120000    2          2         2   26     -1      2      0      0   
2   3      90000    2          2         2   34      0      0      0      0   
3   4      50000    2          2         1   37      0      0      0      0   
4   5      50000    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...          0          0          0         0       689         0   
1  ...       3272       3455       3261         0      1000      1000   
2  ...      14331      14948      15549      1518      1500      1000   
3  ...      28314      28959      29547      2000      2019      1200   
4  ...      20940      19146      19131      2000     36681     10000   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  dpnm  
0         0         0         0     1  
1     

In [16]:
#Data Preprocessing
print(df.isnull().sum())

df.fillna(0, inplace=True)

print("Missing Values:",
      df.isnull().sum().sum())

print("Duplicates:",
      df.duplicated().sum())

df.drop_duplicates(inplace=True)

if "ID" in df.columns:
    df.drop("ID", axis=1, inplace=True)

ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
dpnm         0
dtype: int64
Missing Values: 0
Duplicates: 0


In [17]:
#Feature Engineering
df["TOTAL_BILL"] = (
    df["BILL_AMT1"] +
    df["BILL_AMT2"] +
    df["BILL_AMT3"] +
    df["BILL_AMT4"] +
    df["BILL_AMT5"] +
    df["BILL_AMT6"]
)

df["TOTAL_PAYMENT"] = (
    df["PAY_AMT1"] +
    df["PAY_AMT2"] +
    df["PAY_AMT3"] +
    df["PAY_AMT4"] +
    df["PAY_AMT5"] +
    df["PAY_AMT6"]
)

df["PAYMENT_RATIO"] = (
    df["TOTAL_PAYMENT"] /
    (df["TOTAL_BILL"] + 1)
)

df["AVG_DELAY"] = (
    df["PAY_1"] +
    df["PAY_2"] +
    df["PAY_3"] +
    df["PAY_4"] +
    df["PAY_5"] +
    df["PAY_6"]
)/6

Model Building

In [18]:
#Split dataset
X = df.drop("dpnm", axis=1)
y = df["dpnm"]

In [19]:
#Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
#Handle Imbalance
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [21]:
#Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_smote
)



In [22]:
from xgboost import XGBClassifier

base_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

base_xgb.fit(
    X_train_scaled,
    y_train_smote
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [23]:
# Fine Tuning XGBoost


xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(
    X_train_scaled,
    y_train_smote
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [24]:
#Evaluate Models
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

In [25]:
y_pred_xgb = xgb.predict(
    X_train_scaled
)

print("XGBOOST")

print(
    "Accuracy:",
    accuracy_score(
        y_train_smote,
        y_pred_xgb
    )
)

print(
    classification_report(
        y_train_smote,
        y_pred_xgb
    )
)

cm_xgb = confusion_matrix(
    y_train_smote,
    y_pred_xgb
)

print(cm_xgb)

y_prob_xgb = xgb.predict_proba(
    X_train_scaled
)[:,1]

auc_xgb = roc_auc_score(
    y_train_smote,
    y_prob_xgb
)

print("ROC AUC:", auc_xgb)

XGBOOST
Accuracy: 0.9332031458990958
              precision    recall  f1-score   support

           0       0.91      0.96      0.93     18691
           1       0.96      0.91      0.93     18691

    accuracy                           0.93     37382
   macro avg       0.93      0.93      0.93     37382
weighted avg       0.93      0.93      0.93     37382

[[17936   755]
 [ 1742 16949]]
ROC AUC: 0.9854555048787392


In [26]:
from sklearn.metrics import accuracy_score, roc_auc_score

# Probabilities
base_prob = base_xgb.predict_proba(X_train_scaled)[:, 1]

# Convert probability → class prediction
base_pred = (base_prob >= 0.5).astype(int)

# ROC AUC
base_auc = roc_auc_score(y_train_smote, base_prob)

print("Base Accuracy:",
      accuracy_score(y_train_smote, base_pred))

print("Base ROC AUC:", base_auc)

Base Accuracy: 0.8680648440425873
Base ROC AUC: 0.9432064911927986


In [27]:
results = pd.DataFrame({
    "Model":[
        "XGBoost"
    ],
    "Accuracy":[
        accuracy_score(y_train_smote, y_pred_xgb)
    ],
    "ROC_AUC":[
        auc_xgb
    ]
})

print(results)

     Model  Accuracy   ROC_AUC
0  XGBoost  0.933203  0.985456


In [28]:
import joblib

joblib.dump(
    xgb,
    "credit_default_model.pkl"
)

joblib.dump(
    scaler,
    "scaler.pkl"
)

['scaler.pkl']

In [29]:
sample = X.iloc[[0]]

sample_scaled = scaler.transform(
    sample
)

prediction = xgb.predict(
    sample_scaled
)[0]

probability = xgb.predict_proba(
    sample_scaled
)[0][1]

In [30]:
if probability >= 0.70:
    risk_level = "HIGH"

elif probability >= 0.40:
    risk_level = "MEDIUM"

else:
    risk_level = "LOW"

In [31]:
print("Prediction:", prediction)
print("Probability:", probability)
print("Risk Level:", risk_level)

Prediction: 1
Probability: 0.7522794
Risk Level: HIGH


In [32]:
X_scaled = scaler.transform(X)

predictions = xgb.predict(X_scaled)

probabilities = xgb.predict_proba(
    X_scaled
)[:,1]

In [33]:
risk_levels = []

for prob in probabilities:

    if prob >= 0.70:
        risk_levels.append("HIGH")

    elif prob >= 0.40:
        risk_levels.append("MEDIUM")

    else:
        risk_levels.append("LOW")

In [34]:
results = pd.DataFrame({
    "Predicted_Default": predictions,
    "Risk_Probability": probabilities,
    "Risk_Level": risk_levels
})

results.to_csv(
    "customer_risk_report.csv",
    index=False
)